# The canonical PyTorch training loop

Write the loop every PyTorch project has: batches in, `zero_grad`, forward, loss, `backward`, `step`, with the bookkeeping done right.

Write `train_regressor(X, Y, W0, b0, lr, orders, batch_size)`:

- `X`: float64 `(N, D)`, `Y`: float64 `(N, K)` targets.
- `W0`: `(K, D)`, `b0`: `(K,)`: initial weights of an `nn.Linear(D, K)` (already in PyTorch's `(out, in)` layout).
- `lr`: float, `batch_size`: int $\ge 1$.
- `orders`: int64 `(E, N)`. Row `e` is the sample order for epoch `e` (a permutation of `0..N-1`), so there is no hidden randomness.

## The loop
For each epoch `e`, walk `orders[e]` in consecutive chunks of `batch_size` (the last chunk may be smaller; keep it). For each chunk:

1. forward the batch through the model,
2. loss = `nn.MSELoss()` (mean over all `batch * K` entries),
3. `zero_grad`, `backward`, then one `torch.optim.SGD(lr=lr)` step (no momentum, no weight decay).

## Return
A tuple `(epoch_loss, W, b)`:

- `epoch_loss`: float64 `(E,)`. For epoch `e`, the average over **samples** of the loss seen during that epoch: $\sum_{\text{batches}} \text{loss}_\beta \cdot |\beta| \,/\, N$, using each batch's loss before its update.
- `W`, `b`: the final weights.

Don't modify the input tensors.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
from torch import nn

def train_regressor(X, Y, W0, b0, lr, orders, batch_size):
    """Return (epoch_loss (E,), W (K, D), b (K,))."""
    model = nn.Linear(X.shape[1], Y.shape[1]).double()
    with torch.no_grad():
        model.weight.copy_(W0)
        model.bias.copy_(b0)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    epoch_loss = []
    for order in orders:
        total = 0.0
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            loss = nn.functional.mse_loss(model(X[idx]), Y[idx])
            loss.backward()
            opt.step()
            total += loss.item()
        epoch_loss.append(total / len(order))
    return torch.tensor(epoch_loss), model.weight.detach(), model.bias.detach()


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0], [2.0]], "dtype": "float64"}, {"__nd__": [[2.0], [4.0]], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, 0.1, {"__nd__": [[0, 1]], "dtype": "int64"}, 1], "e": {"__tup__": [{"__nd__": [5.92], "dtype": "float64"}, {"__nd__": [[1.52]], "dtype": "float64"}, {"__nd__": [0.96], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [2.0], [3.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.5], "dtype": "float64"}, 0.05, {"__nd__": [[2, 0, 1], [1, 2, 0]], "dtype": "int64"}, 2], "e": {"__tup__": [{"__nd__": [2.6668749999999997, 1.4443900833333332], "dtype": "float64"}, {"__nd__": [[0.22605, 0.388]], "dtype": "float64"}, {"__nd__": [0.8975500000000001], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.101262453505847, 0.3384312766461778, -0.5399715152535035], [-1.2602418568524327, -1.8946212698392553, 0.018638290983285614], [-0.8105670995116028, -0.8721559599345132, -0.22196950708389104], [-0.05184602813201771, -2.2767828157758307, 0.9251465905764266], [-2.026845605910026, 1.8596241085543468, 0.5905681115998624], [-0.47184451813331413, 1.342996537380774, 0.020786461917443023], [0.6909209676232841, 0.10722707944720507, 1.096531380130275], [1.0614224543758295, -0.9073341413930657, -0.6124917784436158], [0.3432270521124315, -0.21001015163517062, -2.284914216554451], [2.0261175349217764, -2.174977373868806, -2.081542853744764]], "dtype": "float64"}, {"__nd__": [[-0.6230232335835555, 0.4567123314139157], [-1.91583936995506, -0.21875984087220587], [-0.405963082513215, -0.5346737543929067], [-4.395795523641477, 0.6838795815591658], [5.751555670803452, -1.561558078279699], [2.8732326863897266, -0.6741421737273202], [-1.063136142612518, 0.4705295588082291], [-2.8209605614553355, 0.8210092452390919], [-0.00021472376140953264, -0.2439014130740736], [-5.686919807322591, 1.314355117138915]], "dtype": "float64"}, {"__nd__": [[-0.13150584358653278, 0.2903581786972847, -0.17381109288833918], [0.16042742994540782, -0.08970721577902044, -0.011561548050669125]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 0.05, {"__nd__": [[6, 8, 1, 2, 3, 0, 4, 9, 5, 7], [7, 1, 2, 0, 6, 8, 5, 4, 9, 3], [1, 4, 0, 8, 3, 7, 6, 2, 9, 5]], "dtype": "int64"}, 4], "e": {"__tup__": [{"__nd__": [3.8177682929817394, 1.5790836557289987, 0.6289565535546205], "dtype": "float64"}, {"__nd__": [[-0.811147811453096, 1.3867323474770556, 0.1076024511146046], [0.333757939897392, -0.23820068758236976, -0.023518423632168876]], "dtype": "float64"}, {"__nd__": [-0.17961600656020338, -0.008421761978121044], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.06555257091474957, -0.6522226650767547], [2.2283926348223857, 0.09491534615827704], [-1.3210737429953536, 0.6465603079512943], [0.7124017466131326, -1.1474830383225882], [0.5356130419751398, 0.06895578792877383], [0.8203356602681026, 0.7121954717945739], [-0.008442406578893256, -0.4082828680274049]], "dtype": "float64"}, {"__nd__": [[-0.41288062673385817], [2.0039082755581856], [-1.0484985141008976], [0.2629448647149614], [0.7376335864978665], [0.9321415005211485], [-0.08782593907047122]], "dtype": "float64"}, {"__nd__": [[-0.11658908353948441, -0.011337249571100234]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, 0.1, {"__nd__": [[3, 4, 2, 1, 0, 5, 6], [0, 3, 6, 4, 1, 5, 2]], "dtype": "int64"}, 7], "e": {"__tup__": [{"__nd__": [1.224981332096979, 0.7039199395681308], "dtype": "float64"}, {"__nd__": [[0.28939430199758764, 0.0048063616046109875]], "dtype": "float64"}, {"__nd__": [0.12052221660927222], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.398012033385506, -0.22641575599886163, 1.148429491914847, 0.41259870912035135], [-0.3088970300382739, -2.6333949121689955, 0.22213365595929296, -0.2769451411638737], [-1.600519256790602, 0.5145047983461842, -0.02542251765492415, 0.37505307082165046], [-0.6044158958807989, 0.20202256113721573, -0.4732319360358482, 1.064946673093174], [0.12432566647692697, -1.4025242817438563, 0.21431604721707925, 0.06764111529020526], [-0.20344439907432663, -1.3496622466149306, 1.0595377801274997, -0.7315969199565924], [1.5826354419474944, -0.6599439821810752, 0.03247685935418113, -0.04481292044215691], [0.15117442759971664, 2.6434150100419798, -0.4982878044706027, 2.06945735394623], [0.6234582241749037, -0.6950075410488258, 1.2555604608850626, 0.840824406337222]], "dtype": "float64"}, {"__nd__": [[0.7241615902688907, 0.325564556280867, 1.2434495428940255], [1.2351954727155037, 0.8899810643332675, 0.29627935569109687], [-2.3448361771384523, 2.4518261454742736, -1.0320805325086466], [-2.5451325181965996, 1.3947946609473234, -0.44928159544461055], [0.5114270602407796, 0.3701781662177368, 0.15766624564993625], [1.6240764532892291, 1.2291864099304224, 0.5729287320816752], [1.9054699565593582, -1.8611349563969073, 0.7899032574951587], [-4.631980789114344, 1.168760618273073, -0.13205208273827174], [-0.5024159298203227, 2.315953725346989, 0.9671432326486185]], "dtype": "float64"}, {"__nd__": [[-0.0020842552616392672, -0.16118957448447263, -0.02181032070287421, -0.02197541451213531], [-0.04003641887178976, -0.019982960153146878, -0.0674424582213208, 0.05107633616456997], [-0.0003416785615325009, -0.04715022769968502, -0.02292263750771933, -0.06770607435117254]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}, 0.02, {"__nd__": [[5, 4, 8, 2, 1, 0, 6, 7, 3], [1, 5, 0, 2, 7, 4, 6, 3, 8], [1, 5, 0, 6, 2, 7, 4, 8, 3], [7, 6, 5, 4, 1, 8, 0, 2, 3]], "dtype": "int64"}, 1], "e": {"__tup__": [{"__nd__": [2.044125605139888, 1.4541454299332246, 1.0869001648799994, 0.8484219026730259], "dtype": "float64"}, {"__nd__": [[0.30447116114284317, -0.7077060518387834, 0.13602847294150572, -0.5213952102995442], [-0.3175015491493531, -0.04022034374550651, 0.11183297731990374, 0.28407267138635556], [0.22024561578318033, -0.12538990149125182, 0.11384133843488413, -0.03845901716622914]], "dtype": "float64"}, {"__nd__": [-0.22430898908267516, 0.33105485927447, 0.08631805627852865], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.12750468991531275, -0.3133682396647936, -1.8016334552714692, 1.647543919400945, -0.4449818634321405], [-0.6714382526761568, 0.3004317510791222, -2.089249161034092, -0.39783877576361315, 0.3063333668481765], [-0.2647222050106857, 0.06380195633759325, -0.44448652678519246, -0.36857476773587067, -0.04086542955755042], [0.151554193273145, -1.0952774823973215, 0.2788580234752611, 0.29731142812037387, -0.6188655552848181], [-0.573031058714407, 0.11682139902369978, 1.1628276325849503, 1.0874271231648798, -0.8321997633261109], [-0.5420424975374208, -0.5625668524331573, 0.3804322868908683, -0.16157572450625496, 0.31711213813915573], [1.2430534088452556, 0.20947163269473873, 0.3924898694217761, 0.33192884863050404, -0.7829503273163331], [-0.17699268282934663, -0.26031719929824393, -0.5396404057117445, 1.4791362661993346, 0.5279582962435085], [-2.7235073146238693, -0.8548788935345482, -0.1381025086809192, -0.044827812846135816, -0.8665165908342006], [-0.37104002159794913, -0.4370601725694087, -0.5143146047155737, 0.5874951411009339, -0.09493658034216755], [-0.6114396483141783, -0.3383567058115522, -0.04700391080818494, -1.1425493164092433, -0.5926978992316189], [0.5637663767963987, 0.9575148967776183, -2.3905831806923263, 2.4201542113291676, -0.6503330472862269], [0.7853259352895292, -1.378001956037206, 0.6908892547969407, 0.02559770782155764, -0.0800943571183666], [-0.30010142418165453, -0.2529018642584984, 0.7367587777373799, -0.7547009267105697, -0.2550227778748648], [-0.3948318069452478, -1.6352904120607807, 0.09352351648286154, -0.038902375054806385, -0.4810858401814653], [-1.367976003986649, -0.3494670858987591, -0.16312948980393505, -0.010201615660946052, 0.051451314536709175]], "dtype": "float64"}, {"__nd__": [[0.1866040121382877, -1.0271064724440444], [-0.14608509808977913, -3.5848056877592276], [-0.18155064915331307, -1.47213643957168], [-0.9636091958665837, -0.171836439250191], [1.9334732052293087, 1.953451736755144], [0.2902538842254023, -0.4740087167881992], [-1.3916117410872588, 2.1854349551050087], [0.9447266278360469, 0.47670111900189394], [2.2084777388417853, -3.906799696593852], [-0.014107073061295439, -0.814509173648977], [-0.7231639841748301, -2.2228746595319206], [0.525736783143277, 0.7075675780289958], [-1.7005944352933027, 0.8312727362120743], [-0.26241594430759485, -0.5267390870450923], [-0.977342013827147, -1.6824297957395373], [1.3124304668387985, -2.0450258923075957]], "dtype": "float64"}, {"__nd__": [[-0.14870404743423105, 0.04726766478848071, -0.19848406537946217, 0.07406110051447348, -0.11881482789828444], [0.14472517463740706, -0.06468294244697202, 0.016044083545189332, 0.031411877427508984, 0.0021568702034596775]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 0.1, {"__nd__": [[9, 5, 15, 2, 0, 10, 4, 8, 12, 1, 3, 11, 6, 13, 14, 7], [3, 11, 5, 0, 4, 6, 13, 9, 10, 14, 15, 7, 8, 12, 1, 2], [11, 4, 12, 13, 2, 0, 10, 5, 15, 1, 9, 7, 14, 6, 8, 3], [15, 2, 10, 3, 14, 1, 0, 11, 12, 8, 5, 6, 4, 9, 13, 7], [9, 12, 15, 7, 11, 0, 8, 13, 4, 10, 2, 3, 14, 6, 1, 5]], "dtype": "int64"}, 5], "e": {"__tup__": [{"__nd__": [1.8075902205373215, 1.0633449846875855, 0.642456305566019, 0.44469884810592186, 0.31882063963470075], "dtype": "float64"}, {"__nd__": [[-0.7754529766897509, 0.285014964899642, 0.010699539210288388, 0.3934094124688081, 0.027647417156347574], [1.022753095223354, 0.23141586260239197, 0.740809941308548, 0.7149724475267653, 0.03713400979153234]], "dtype": "float64"}, {"__nd__": [-0.10708709163865814, -0.2852508415062334], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0], [1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [[3.0], [3.0], [3.0], [3.0]], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, 0.0, {"__nd__": [[3, 2, 1, 0]], "dtype": "int64"}, 3], "e": {"__tup__": [{"__nd__": [9.0], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(train_regressor, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")